# MIDAS functional test — MidasCivil resilience, triage & staged-model results

Live functional exercise of `civilpy.structural.midas.MidasCivil` against a
running **Civil NX** session. Run this **from the civilpy repo root** — it
imports `src.civilpy` directly, so no install/release is needed; edit the source
and re-run.

What it checks (driven by a 23-model production batch):

1. **Triage** — `summarize()` / `capability()` decide whether each model can be
   analyzed before spending a solve on it.
2. **Resilient batch** — typed errors (`MidasConnectionError` / `MidasTimeoutError`
   / `MidasLicenseError`); a dropped session pauses the run instead of emitting an
   identical error for every remaining model.
3. **Staged-model result discovery** — a construction-stage model keeps its
   results under load **combinations** (`(CB)`), *not* the static `(ST)` cases.
   The deck girder (`CUY-71`) solved but returned 0 beam-force rows for exactly
   this reason; here we try combos → static → moving and report which yields rows.

Self-contained: no snbi_ui / AASHTO imports (civilpy is public). Set `MCB_DIR`
below to the folder of `.mcb` files and open Civil NX first.

In [ ]:
import os, sys, json, glob, re, pathlib
from collections import Counter

sys.path.insert(0, os.path.abspath("."))      # run from the civilpy repo root
from src.civilpy.structural.midas import (
    MidasCivil, MidasApiError, MidasConnectionError,
    MidasTimeoutError, MidasLicenseError,
    parse_result_table, envelope)              # result-table helpers live in civilpy now

def structure_type(path):
    m = re.search(r"\(([^)]+)\)", os.path.basename(path))
    return m.group(1) if m else "?"

# --- connect -----------------------------------------------------------------
midas = MidasCivil()           # analysis_timeout / reconnect_retries = defaults
LIVE = midas.ping()
print(midas, "| live:", LIVE)

MCB_DIR = os.environ.get("MCB_DIR", r"")      # <- set to your Flattened_MCBs folder
MODELS = sorted(glob.glob(os.path.join(MCB_DIR, "*.mcb"))) if MCB_DIR else []
print(len(MODELS), "models in", MCB_DIR or "(set MCB_DIR)")


## §1. Load-case discovery — staged models keep results under combinations

`beam_forces()` needs the *result* load-case names. For a plain model those are
the static cases `Name(ST)` and moving loads `Name(MV:all)`. For a
**construction-stage** model the static cases aren't retained post-solve — the
results live under the load **combinations** `Name(CB)` (e.g. `DL_SerII_CS(CB)`).
We try the widest-likely set first and report which one returns rows.

In [ ]:
def ids_and_cases(midas):
    """Beam element ids + candidate result load-case names from the open model."""
    elem = midas.elements().get("ELEM", {})
    ids = sorted(int(i) for i, e in elem.items()
                 if e.get("TYPE") == "BEAM" and str(i).isdigit())[:400]
    stld = midas.static_loads().get("STLD", {})
    mvld = midas.get_db("MVLD").get("MVLD", {})
    lcom = midas.load_combinations().get("LCOM-GEN", {})
    static = [v["NAME"] for v in stld.values() if v.get("NAME")][:4]
    moving = [(v.get("LCNAME") or v.get("NAME")) for v in mvld.values()
              if (v.get("LCNAME") or v.get("NAME"))][:3]
    combos = [v["NAME"] for v in lcom.values() if v.get("NAME")][:4]
    return ids, static, moving, combos

def candidate_case_sets(static, moving, combos):
    """Result-case name candidates, widest-first. Combos (CB) lead because a
    staged model only retains results there (grounded in CUY-71 deck girder:
    its only results were DL_SerII_CS / DW_SerII_CS combinations)."""
    sets = []
    if combos or moving:
        sets.append(("combos(CB)+moving(MV)",
                     [f"{c}(CB)" for c in combos] + [f"{m}(MV:all)" for m in moving]))
    if static or moving:
        sets.append(("static(ST)+moving(MV)",
                     [f"{s}(ST)" for s in static] + [f"{m}(MV:all)" for m in moving]))
    if combos:
        sets.append(("combos(CB)", [f"{c}(CB)" for c in combos]))
    return sets

def extract_with_discovery(midas, ids, static, moving, combos):
    """Try each candidate case set until beam_forces returns rows."""
    for label, cases in candidate_case_sets(static, moving, combos):
        if not cases or not ids:
            continue
        try:
            resp = midas.beam_forces(ids, cases)   # validate=True warns on aliasing
        except MidasApiError:
            continue
        rows = parse_result_table(resp)
        if rows:
            return rows, label
    return [], None

## §2. Resilient batch — open → triage → analyze → extract

One pass per model: open it, `summarize()`/`capability()` to decide if it's worth
solving, then `analyze()` and pull beam forces with case discovery. Typed errors
are classified; a confirmed connection drop **pauses** the batch (the rest are
left un-run for a re-run) rather than failing every remaining model.

In [ ]:
def run_model(midas, path):
    out = {"file": os.path.basename(path), "type": structure_type(path)}
    try:                                       # a 400 here is a license/tier limit
        midas.open(path)
    except MidasLicenseError as e:
        out["error_kind"], out["error"] = "license", str(e)[:140]; return out
    except MidasConnectionError as e:
        out["error_kind"], out["error"] = "connection", str(e)[:140]; return out
    except MidasApiError as e:
        out["error_kind"], out["error"] = "open", str(e)[:140]; return out
    s = midas.summarize()
    can, ll, note = midas.capability(summary=s)
    out.update(elems=s["elems"], elem_types=s["elem_types"],
               can_analyze=can, can_rate_ll=ll, note=note)
    if not can:
        out["skipped"] = True; return out
    try:                                       # civilpy uses its long analysis_timeout
        midas.analyze()
    except MidasTimeoutError as e:
        out["error_kind"], out["error"] = "timeout", str(e)[:140]; return out
    except MidasConnectionError as e:
        out["error_kind"], out["error"] = "connection", str(e)[:140]; return out
    except MidasApiError as e:
        out["error_kind"], out["error"] = "analyze", str(e)[:140]; return out
    ids, static, moving, combos = ids_and_cases(midas)
    out["n_beam_elems"] = len(ids)
    rows, label = extract_with_discovery(midas, ids, static, moving, combos)
    out["beamforce_query"] = label
    out["beamforce_rows"] = len(rows)
    if rows:
        out["max_My"]    = envelope(rows, "Moment-y")
        out["max_Vz"]    = envelope(rows, "Shear-z")
        out["max_Axial"] = envelope(rows, "Axial")
    elif static or moving or combos:
        out["beamforce_note"] = "0 rows after trying combo(CB)/static(ST)/moving(MV)"
    return out

if LIVE and MODELS:
    res_dir = pathlib.Path("midas_func_results"); res_dir.mkdir(exist_ok=True)
    results, aborted = [], []
    for idx, path in enumerate(MODELS):
        try:
            r = run_model(midas, path)
        except Exception as e:                 # never let one model halt the batch
            r = {"file": os.path.basename(path), "error": f"unexpected: {str(e)[:140]}"}
        (res_dir / (pathlib.Path(path).stem + ".json")).write_text(json.dumps(r, indent=1))
        results.append(r)
        name = r["file"][:32]
        if r.get("error"):
            print(f"  ERR  {r.get('error_kind','?'):10s} {name:32s} — {r['error'][:50]}")
            # civilpy already retried a transient blip; a surviving connection
            # error + a failing ping means Civil NX is actually down — stop.
            if r.get("error_kind") == "connection" and not midas.ping():
                aborted = MODELS[idx + 1:]
                print(f"\n  X Civil NX unreachable — pausing with {len(aborted)} "
                      f"model(s) un-run. Reopen Civil NX and re-run this cell.")
                break
        elif r.get("skipped"):
            print(f"  SKIP            {name:32s} — {r['note']}")
        else:
            tag = "OK " if r.get("beamforce_rows") else "OK?"   # OK? = solved, 0 rows
            print(f"  {tag}             {name:32s} rows={r.get('beamforce_rows')} "
                  f"via {r.get('beamforce_query')} My={r.get('max_My')}")
    solved = [r for r in results if r.get("beamforce_rows")]
    print(f"\n{len(solved)} solved w/ forces, {len(results)} attempted, "
          f"{len(aborted)} un-run. -> midas_func_results/")
else:
    print("OFFLINE or no MCB_DIR — open Civil NX and set MCB_DIR to run live.")

## §3. Rating — MBE §6A.4.2.1 (LRFR)

Turn a controlling demand into an AASHTO LRFR rating factor with civilpy's
`aashto.lrfd.lrfr`. `ll_im` is the live-load force effect *including*
distribution factor and dynamic load allowance (IM). A representative demand is
shown until member capacities are wired in from a section calc — the commented
loop shows how to drive it from the §2 batch envelopes (`max_My` per model).


In [ ]:
from src.civilpy.structural.aashto import lrfd

# Representative moment demands (kip-ft) until capacities are wired in.
Mn, dc, dw, ll = 1200.0, 150.0, 40.0, 300.0
r = lrfd.rating_factor(Mn, dc, ll * 1.33, dw=dw)      # ll_im includes IM (1.33)
print(f"civilpy §{r.article} inventory RF = {r.capacity:.3f} [{'OK' if r.ok else 'NG'}]")

# Drive it from the §2 envelopes once member capacity (Mn per girder) is known:
# for row in results:
#     if row.get("max_My"):
#         rf = lrfd.rating_factor(Mn_girder, dc_My, row["max_My"] * 1.33, dw=dw_My)
#         print(f"  {row['file']:32s} RF={rf.capacity:.2f}")


## §4. Notes

- **OK?** rows mean the model solved but no case set returned forces — inspect
  the model's real result-case names (combinations vs stages) and extend
  `candidate_case_sets`.
- A `UserWarning: duplicate column headers` from `beam_forces` flags the
  Axial/Shear-z aliasing seen on one model — verify those envelopes before use.
- `analysis_timeout=` (e.g. `MidasCivil(analysis_timeout=1800)`) for very large
  solves; `reconnect_retries=` tunes the transient-drop backoff.